# Signal Processing: FIR Filter Design by Optimization

A digital FIR lowpass filter is fully described by its tap coefficients $h = (h_0, \ldots, h_{N-1})$. Instead of the closed-form windowed-sinc design (`scipy.signal.firwin`), we can treat tap design as a direct **continuous optimization problem**: minimize the mean-squared error between the filter's actual frequency response and an ideal brick-wall response, over a grid of frequencies:

$$\min_{h \in \mathbb{R}^N} \ \frac{1}{K}\sum_{k=1}^{K} \left(|H(e^{j\omega_k}, h)| - D(\omega_k)\right)^2$$

This is a genuinely higher-dimensional problem (here $N=21$) — a good check that `ParticleSwarmOptimization.optimize(objective_fn, bounds)` generalizes beyond the 2D benchmark landscapes in the other notebooks.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import plotly.graph_objects as go
from scipy import signal

from metaheuristics.algorithms.differential_evolution import DifferentialEvolution
from metaheuristics.algorithms.particle_swarm import ParticleSwarmOptimization

In [3]:
NUM_TAPS = 21
CUTOFF = 0.3  # normalized frequency, Nyquist = 1
FREQS = np.linspace(0, 1, 256)
DESIRED = (FREQS <= CUTOFF).astype(float)

def filter_error(taps):
    _, h = signal.freqz(taps, worN=FREQS * np.pi)
    return float(np.mean((np.abs(h) - DESIRED) ** 2))

bounds = [(-1.0, 1.0)] * NUM_TAPS

## Optimize with PSO and DE, compare to the closed-form `firwin` design

In [4]:
np.random.seed(0)
pso_result = ParticleSwarmOptimization(num_particles=60, max_iterations=200).optimize(filter_error, bounds)
np.random.seed(0)
de_result = DifferentialEvolution(population_size=60, max_generations=150).optimize(filter_error, bounds)

taps_firwin = signal.firwin(NUM_TAPS, CUTOFF)

print(f'PSO    MSE: {pso_result.best_fitness:.5f}')
print(f'DE     MSE: {de_result.best_fitness:.5f}')
print(f'firwin MSE: {filter_error(taps_firwin):.5f}')

PSO    MSE: 0.00879
DE     MSE: 1.17000
firwin MSE: 0.01691


`firwin` isn't actually minimizing this squared error — it optimizes a windowed-sinc criterion instead, which tends to generalize better (lower stopband ripple) even when its MSE on this exact frequency grid looks worse. The optimizers are overfitting directly to the sampled grid `FREQS`.

In [5]:
def response(taps):
    _, h = signal.freqz(taps, worN=FREQS * np.pi)
    return np.abs(h)

fig = go.Figure()
fig.add_trace(go.Scatter(x=FREQS, y=DESIRED, name='ideal', line={'dash': 'dash', 'color': 'black'}))
fig.add_trace(go.Scatter(x=FREQS, y=response(pso_result.best_solution), name='PSO'))
fig.add_trace(go.Scatter(x=FREQS, y=response(de_result.best_solution), name='DE'))
fig.add_trace(go.Scatter(x=FREQS, y=response(taps_firwin), name='firwin'))
fig.update_layout(title='FIR lowpass frequency response', xaxis_title='normalized frequency', yaxis_title='|H|')
fig.show()